# Laya V3.5 — Fused Triton CUDA: Replace ALL Full-Attention Layers

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V35_AllFullAttention_FusedTriton_CUDA_Colab.ipynb)

## Objective
**Compile the best V3.4 architecture into CUDA kernels and replace every ModernBERT full-attention layer** of pretrained \`convaiinnovations/laya-typed-decisions\`; preserve the original pretrained Q/K/V, RoPE, Wₒ and *every sliding-attention layer*.

V3.4 showed \`local256_remote8, route_blocks=8\` was best in all **10** full-attention layers. On 120 decisions, all-layer agreement was **94.17%** and gold accuracy matched the teacher at **82.5%**, but the Python implementation took **1,281 ms** vs **91 ms** native. These are **prior V3.4 measurements**, not V3.5 performance claims.

### GPU architecture
- **Triton kernel 1**: per (batch, head, 16-token block), build *five pretrained key prototypes*: masked mean, first, middle, last, largest key norm. Precomputed once per forward.
- **Triton kernel 2**: one fused GPU program per query/head. Score blocks using the five prototypes, choose **eight remote blocks**, compute exact pretrained Q/K scores for their 128 original tokens, choose **eight best remote keys**, gather original **256 local + eight remote K/V**, and apply **one unified, numerically stable softmax and value reduction**. **No Python query loop or full T×T score matrix in sparse inference.**
- QKV projection + RoPE + output projection reuse pretrained PyTorch modules; their GEMMs are already highly optimized. Triton JIT compiles the custom routing/selection/softmax kernels to GPU machine code on first invocation and caches compilations per shape/dtype.
- **Important:** This is *sparse attention, not attention-free CeNN*. It exactly implements the V3.4 **heuristic** retrieval design, not V3.3's teacher-informed oracle.

### Evaluation and safety
- CUDA/FP16/BF16 preflight at multiple lengths, with boolean/additive 4D masks and masked padding; compare fused CUDA to exact native SDPA for lengths ≤256 and to V3.4's unfused Python reference at longer lengths. Fail before evaluation if the kernel disagrees.
- Benchmark **native ModernBERT**, **V3.4 Python sparse**, and **V3.5 Triton sparse** on identical layer shapes. Only actual GPU timings count.
- Replace **all full-attention layers** simultaneously in a fresh pretrained model, preserving sliding attention. Measure cumulative downstream agreement, gold accuracy, JS divergence, and end-to-end latency. A separate **optional native-fallback model** can pick native attention where it is faster; its results must be labeled separately and never confused with the all-sparse result.
- Audit samples drawn from the dataset TRAIN split, held out by **case**, no official TEST split. All candidates use frozen original weights; training steps = **zero**.
- Real acceptance: minimum layer cosine 0.95, full-model teacher agreement 0.95, ≤1 pp gold-accuracy loss, **and faster than native**. The experiment may fail on short 512-token sequences because optimized native SDPA is excellent.
- No claim of a speedup until this Colab is run on an actual GPU. GPU compilation and benchmarking **cannot** be done in the creation environment.


In [ ]:
#@title 1. Install pinned Laya dependencies + inspect CUDA and Triton
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"
# Triton usually comes with Colab's PyTorch wheel. Use its compatible version.
import torch
try:
    import triton
except ImportError as err:
    raise RuntimeError(
        "CUDA Triton is required. Install a torch-compatible Triton wheel "
        "(for example %pip install triton), restart runtime and rerun."
    ) from err
print("Torch:",torch.__version__,"CUDA:",torch.version.cuda,
      "Triton:",triton.__version__)
assert torch.cuda.is_available(),"Select Runtime → Change runtime type → GPU"
print("GPU:",torch.cuda.get_device_name(0))


In [ ]:
#@title 2. Imports, frozen Laya teacher and experiment configuration
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from tinycenn_lm.decision_v25 import layer_kinds, MultiscaleCeNN
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence,
    collate_items, normalize_question,
)
from tinycenn_lm.decision_training import require_finite

SEED = 2035
SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}
LAYER = 15 #@param {type:"integer"}
TARGET_LAYERS = (LAYER,)
TRAIN_CASES = 100 #@param {type:"integer"}
DEV_CASES = 160 #@param {type:"integer"}
MAX_LEN = 512 #@param {type:"integer"}
BATCH = 3 #@param {type:"integer"}
STEPS = 400 #@param {type:"integer"}
REFINE_STEPS = 1200 #@param {type:"integer"}
REFINE_TOP_K = 2 #@param {type:"integer"}
TRAIN_CACHE_BATCHES = 1 #@param {type:"integer"}
PROBE_CACHE_BATCHES = 2 #@param {type:"integer"}
AUDIT_CACHE_BATCHES = 3 #@param {type:"integer"}
CHECK_EVERY = 100 #@param {type:"integer"}
OUTPUT_DIR = Path("/content/Laya_V35_FusedTriton_AllFull")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert DEV_CASES >= 4 and TRAIN_CACHE_BATCHES > 0
assert REFINE_TOP_K >= 1 and TRAIN_CACHE_BATCHES > 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Use a CUDA GPU runtime in Google Colab.")
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

source_dir = Path(snapshot_download(
    SOURCE_MODEL, allow_patterns=[
        "rl_agent_config.json", "model.safetensors", "encoder/*", "tokenizer/*",
    ],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
student_eval, _ = build_checkpoint_model(source_dir)
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))
teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile = False
student_eval.encoder.config.reference_compile = False

kinds = layer_kinds(teacher.encoder)
full_layers = [i for i, kind in enumerate(kinds) if kind == "full_attention"]
assert LAYER in full_layers, (LAYER, full_layers)
print("GPU:", torch.cuda.get_device_name(0), "AMP:", amp_dtype)
print("Source:", SOURCE_MODEL)
print("Target:", LAYER, "all full attention layers:", full_layers)
# V3.5 compiled CUDA controls
BLOCK_SIZE = 16
QUERY_CHUNK = 32  # V3.4 Python-reference only; NO query chunks in fused kernel
ROUTE_BLOCKS = 8
AUDIT_DECISIONS = 200 #@param {type:"integer"}
CUMULATIVE_E2E_DECISIONS = 120 #@param {type:"integer"}
RUN_ALL_FULL_LAYERS = True #@param {type:"boolean"}
BENCHMARK_PYTHON_REFERENCE = True #@param {type:"boolean"}
RUN_NATIVE_FALLBACK_COMPARISON = True #@param {type:"boolean"}
MAX_MODEL_AUDIT_BATCHES = 4 #@param {type:"integer"}
assert BLOCK_SIZE == 16 and ROUTE_BLOCKS == 8
print("All pretrained full-attention layers:",full_layers)
print("V3.5: compiled Triton CUDA routing + all-layer replacement; no training.")


In [ ]:
#@title 3. Leak-free train/dev data and teacher I/O helpers
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def _snapshot_attention_argument(value):
    """Retain the exact forward inputs for ModernBERT SDPA replay/timing."""
    if torch.is_tensor(value):
        return value.detach()
    if isinstance(value, tuple):
        return tuple(_snapshot_attention_argument(v) for v in value)
    if isinstance(value, list):
        return [_snapshot_attention_argument(v) for v in value]
    if isinstance(value, dict):
        return {k: _snapshot_attention_argument(v) for k, v in value.items()}
    return value

def _attention_output_tensor(result):
    """ModernBERT variants return either tensor or (tensor, optional weights)."""
    return result[0] if isinstance(result, (tuple, list)) else result

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        # The encoder passes more than x/position_embeddings/attention_mask:
        # position_ids, sliding_window_mask and optional flags are required by
        # transformers 4.57.6 SDPA. Never infer these for benchmarking.
        got["call_args"] = _snapshot_attention_argument(tuple(args))
        got["call_kwargs"] = _snapshot_attention_argument(dict(kwargs))
        got["x"] = args[0].detach()
        got["pos"] = _snapshot_attention_argument(kwargs.get("position_embeddings"))
        got["mask"] = _snapshot_attention_argument(kwargs.get("attention_mask"))
    def post(mod, args, kwargs, out):
        got["y"] = _attention_output_tensor(out).detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_args": got["call_args"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred, _ = repl(
                    entry["x"],
                    position_ids=entry["call_kwargs"].get("position_ids"),
                    attention_mask=entry["mask"],
                )
            require_finite(pred, "V3.5 local probe")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/probe/audit decisions:", len(train_items), len(probe_items), len(audit_items))
print("unique train/probe/audit cases:", len(train_rows), len(probe_rows), len(audit_rows))
assert len(probe_items) >= PROBE_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce PROBE_CACHE_BATCHES"
assert len(audit_items) >= AUDIT_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce AUDIT_CACHE_BATCHES"
print("official test remains untouched")


In [ ]:
#@title 4. Slow V3.4 reference only — for CUDA numerical tests and baseline timing
def python_reference_sparse_core(q,k,v,valid,attention_mask=None,
                      mode="local256_remote8",block=BLOCK_SIZE,
                      route_blocks=4,query_chunk=QUERY_CHUNK,
                      return_indices=False):
    """Select tokens by multi-prototype block screening and shortlisted exact QK.

    Does NOT compute NxN QK anywhere. Query chunks limit gathered K/V memory.
    q, k, v: [B, H, T, Dh], with native RoPE already applied to q and k.
    valid: [B,T] boolean; mask: [B,1,T,T] bool/additive or [B,T].
    """
    B,H,T,D=q.shape
    if mode not in ("local256_remote8","local256_remote16","global_top32"):
        raise ValueError(mode)
    if T<1: raise ValueError("Empty sequence is unsupported")
    G=(T+block-1)//block
    extra=G*block-T
    kp=F.pad(k,(0,0,0,extra)).reshape(B,H,G,block,D).float()
    vp=F.pad(valid,(0,extra),value=False).reshape(B,G,block)
    mask=vp[:,None,:,:,None].float()
    representative_mean=(kp*mask).sum(-2)/mask.sum(-2).clamp_min(1)
    first=kp[:,:,:,0,:]
    valid_count=vp.sum(-1).clamp_min(1)
    center=((valid_count-1)//2)[:,None,:].expand(B,H,G)
    last=(valid_count-1)[:,None,:].expand(B,H,G)
    norms=kp.square().sum(-1).masked_fill(~vp[:,None,:,:],-1e30)
    maximal=norms.argmax(-1)
    def get_representative(which):
        return kp.gather(
            3,which[...,None,None].expand(B,H,G,1,D)
        ).squeeze(3)
    prototypes=torch.stack((representative_mean,first,
                            get_representative(center),
                            get_representative(last),
                            get_representative(maximal)),dim=-2)
    group_valid=vp.any(-1)
    block_start=torch.arange(G,device=q.device)*block
    block_end=torch.minimum(block_start+block-1,
                            torch.tensor(T-1,device=q.device))
    ib=torch.arange(B,device=q.device)[:,None,None,None]
    ih=torch.arange(H,device=q.device)[None,:,None,None]
    chunks=[]
    selected_chunks=[]
    available_chunks=[]
    for start in range(0,T,query_chunk):
        stop=min(T,start+query_chunk)
        C=stop-start
        qc=q[:,:,start:stop].float()
        query_pos=torch.arange(start,stop,device=q.device)
        # Block-ranking cost O(B H T G 5 d); no T-by-T query/key matrix.
        # CUDA autocast implicitly casts einsum to FP16, even when both
        # operands were .float(). FP16 cannot represent the -1e9 mask
        # sentinel and would crash in masked_fill. Keep routing logits FP32.
        with torch.autocast(device_type=q.device.type, enabled=False):
            routed=torch.einsum(
                "bhcd,bhgrd->bhcgr",qc.float(),prototypes.float()
            ).amax(-1)/math.sqrt(D)
        assert routed.dtype==torch.float32
        routed=routed.masked_fill(~group_valid[:,None,None,:],-1e9)
        if mode.startswith("local"):
            far_block=((block_start[None,:]<query_pos[:,None]-127) |
                       (block_end[None,:]>query_pos[:,None]+128))
            routed=routed.masked_fill(~far_block[None,None,:,:],-1e9)
        selected_blocks=torch.topk(routed,k=min(route_blocks,G),dim=-1).indices
        candidate=(selected_blocks[...,None]*block+
                   torch.arange(block,device=q.device)).reshape(B,H,C,-1)
        candidate_in_range=candidate<T
        candidate=candidate.clamp(max=T-1)

        def exact_scores(indices,valid_range=None):
            # Advanced indexing returns ONLY the shortlisted B,H,C,N,D tensors.
            chosen_k=k[ib,ih,indices,:].float()
            chosen_v=v[ib,ih,indices,:].float()
            # Elementwise FP32 scores remain FP32 under CUDA autocast.
            # A future matmul/einsum optimization must disable autocast here.
            scores=(qc[:,:,:,None,:]*chosen_k).sum(-1).float()/math.sqrt(D)
            allowed=valid[:,None,None,:].expand(B,H,C,T).gather(-1,indices)
            if valid_range is not None:
                allowed=allowed & valid_range
            if attention_mask is not None:
                if attention_mask.ndim==4:
                    m4=attention_mask[:,:,start:stop,:]
                    m4=m4.expand(B,H,C,T).gather(-1,indices)
                    if m4.dtype==torch.bool:
                        allowed=allowed & m4
                    else:
                        scores=scores+m4.float()
                        allowed=allowed & (m4>-1e4)
                elif attention_mask.ndim!=2:
                    raise ValueError(f"Unsupported mask dimensionality {attention_mask.ndim}")
            return scores.masked_fill(~allowed,-1e9),allowed,chosen_v

        candidate_scores,candidate_valid,_=exact_scores(candidate,candidate_in_range)
        if mode=="global_top32":
            kept=min(32,candidate.shape[-1])
        else:
            kept=min(8 if mode=="local256_remote8" else 16,candidate.shape[-1])
            truly_remote=((candidate<query_pos[None,None,:,None]-127) |
                          (candidate>query_pos[None,None,:,None]+128))
            candidate_valid=candidate_valid & truly_remote
            candidate_scores=candidate_scores.masked_fill(~candidate_valid,-1e9)
        selected=torch.topk(candidate_scores,k=kept,dim=-1).indices
        distant=candidate.gather(-1,selected)
        distant_valid=candidate_valid.gather(-1,selected)
        if mode.startswith("local"):
            offsets=torch.arange(-127,129,device=q.device)
            local_positions=query_pos[:,None]+offsets[None,:]
            local_ok=(local_positions>=0)&(local_positions<T)
            local_positions=local_positions.clamp(0,T-1)
            near=local_positions[None,None,:,:].expand(B,H,C,256)
            indices=torch.cat((near,distant),dim=-1)
            available=torch.cat(
                (local_ok[None,None,:,:].expand(B,H,C,256),distant_valid),
                dim=-1
            )
        else:
            indices=distant
            available=distant_valid
        logits,original_valid,values=exact_scores(indices,available)
        available=available & original_valid
        logits=logits.masked_fill(~available,-1e9)
        prob=torch.softmax(logits,dim=-1)
        mixed=(prob.unsqueeze(-1)*values).sum(-2).to(q.dtype)
        mixed=mixed*valid[:,None,start:stop,None].to(mixed.dtype)
        chunks.append(mixed)
        if return_indices:
            selected_chunks.append(indices)
            available_chunks.append(available)
    output=torch.cat(chunks,dim=2)
    if return_indices:
        return output,torch.cat(selected_chunks,dim=2),torch.cat(available_chunks,dim=2)
    return output


In [ ]:
#@title 5. TWO compiled Triton CUDA kernels: prototype construction + fused retrieval
import triton
import triton.language as tl
from transformers.models.modernbert.modeling_modernbert import apply_rotary_pos_emb
from tinycenn_lm.laya_lab.core import _valid_tokens

@triton.jit
def _build_key_prototypes(K, VALID, PROTO, GROUP_VALID,
                          T:tl.constexpr,H:tl.constexpr,D:tl.constexpr,
                          G:tl.constexpr,BD:tl.constexpr):
    # CTA = one (batch, head, 16-token key block)
    g=tl.program_id(0)
    h=tl.program_id(1)
    b=tl.program_id(2)
    offs=tl.arange(0,16)
    d=tl.arange(0,BD)
    pos=g*16+offs
    iskey=(pos<T)&(tl.load(VALID+b*T+pos,mask=pos<T,other=0)!=0)
    kval=tl.load(
        K+((b*H+h)*T+pos[:,None])*D+d[None,:],
        mask=(pos[:,None]<T)&(d[None,:]<D),other=0
    ).to(tl.float32)
    kval=tl.where(iskey[:,None],kval,0.)
    count=tl.sum(iskey.to(tl.int32),0)
    mean=tl.sum(kval,0)/tl.maximum(count,1)
    norms=tl.sum(kval*kval,1)
    norms=tl.where(iskey,norms,-1.0e9)
    imax=tl.argmax(norms,0)
    mid=tl.maximum(count-1,0)//2
    last=tl.maximum(count-1,0)
    base=PROTO+((b*H+h)*G+g)*5*D+d
    tl.store(base+0*D,mean,mask=d<D)
    tl.store(base+1*D,tl.sum(tl.where(offs[:,None]==0,kval,0.),0),mask=d<D)
    tl.store(base+2*D,tl.sum(tl.where(offs[:,None]==mid,kval,0.),0),mask=d<D)
    tl.store(base+3*D,tl.sum(tl.where(offs[:,None]==last,kval,0.),0),mask=d<D)
    tl.store(base+4*D,tl.sum(tl.where(offs[:,None]==imax,kval,0.),0),mask=d<D)
    if h==0:
        tl.store(GROUP_VALID+b*G+g,count>0)

@triton.jit
def _fused_local256_remote8(Q,K,V,VALID,PROTO,GROUP_VALID,MASK,OUT,
                            T:tl.constexpr,H:tl.constexpr,D:tl.constexpr,
                            G:tl.constexpr,BD:tl.constexpr,BG:tl.constexpr,
                            HAS_MASK4:tl.constexpr,MASK_IS_BOOL:tl.constexpr,
                            M0:tl.constexpr,M1:tl.constexpr,
                            M2:tl.constexpr,M3:tl.constexpr):
    # CTA = one query and one head; no full T x T query/key score matrix.
    i=tl.program_id(0)
    h=tl.program_id(1)
    b=tl.program_id(2)
    d=tl.arange(0,BD)
    query_valid=tl.load(VALID+b*T+i)!=0
    out_ptr=OUT+((b*H+h)*T+i)*D+d
    if query_valid:
        q=tl.load(Q+((b*H+h)*T+i)*D+d,mask=d<D,other=0).to(tl.float32)
        # Precomputed (mean,first,middle,last,max-norm) key prototypes.
        groups=tl.arange(0,BG)
        rep=tl.arange(0,8)
        p=tl.load(
            PROTO+(((b*H+h)*G+groups[:,None,None])*5+
                   rep[None,:,None])*D+d[None,None,:],
            mask=(groups[:,None,None]<G)&(rep[None,:,None]<5)&(d[None,None,:]<D),
            other=0
        ).to(tl.float32)
        proto_scores=tl.sum(p*q[None,None,:],2)
        proto_scores=tl.where(rep[None,:]<5,proto_scores,-1.0e9)
        ranked=tl.max(proto_scores,1)/tl.sqrt(tl.full((),D,tl.float32))
        gv=tl.load(GROUP_VALID+b*G+groups,mask=groups<G,other=0)!=0
        group_start=groups*16
        group_end=tl.minimum(group_start+15,T-1)
        remote=(group_start<i-127)|(group_end>i+128)
        ranked=tl.where((groups<G)&gv&remote,ranked,-1.0e9)
        # Choose eight remote blocks using one compiled GPU program.
        cand_offset=tl.arange(0,128)
        chosen_blocks=tl.full((128,),0,tl.int32)
        for j in tl.static_range(0,8):
            winner=tl.argmax(ranked,0)
            chosen_blocks=tl.where(cand_offset//16==j,winner,chosen_blocks)
            ranked=tl.where(groups==winner,-1.0e9,ranked)
        candidates=chosen_blocks*16+cand_offset%16
        inrange=candidates<T
        ci=tl.minimum(candidates,T-1)
        key_allowed=tl.load(VALID+b*T+ci,mask=inrange,other=0)!=0
        is_remote=(ci<i-127)|(ci>i+128)
        eligible=inrange&key_allowed&is_remote
        ck=tl.load(
            K+((b*H+h)*T+ci[:,None])*D+d[None,:],
            mask=eligible[:,None]&(d[None,:]<D),other=0
        ).to(tl.float32)
        candidate_scores=tl.sum(ck*q[None,:],1)/tl.sqrt(tl.full((),D,tl.float32))
        if HAS_MASK4:
            mm=tl.load(
                MASK+b*M0+i*M2+ci*M3,
                mask=inrange,other=0
            )
            if MASK_IS_BOOL:
                eligible=eligible&(mm!=0)
            else:
                candidate_scores=candidate_scores+mm.to(tl.float32)
                eligible=eligible&(mm.to(tl.float32)>-1.0e4)
        candidate_scores=tl.where(eligible,candidate_scores,-1.0e9)
        remote_offsets=tl.arange(0,8)
        remote_idx=tl.full((8,),0,tl.int32)
        remote_ok=tl.full((8,),0,tl.int1)
        for j in tl.static_range(0,8):
            best=tl.argmax(candidate_scores,0)
            best_val=tl.max(candidate_scores,0)
            tok=tl.sum(tl.where(cand_offset==best,ci,0),0)
            yes=best_val>-1.0e8
            remote_idx=tl.where(remote_offsets==j,tok,remote_idx)
            remote_ok=tl.where(remote_offsets==j,yes,remote_ok)
            candidate_scores=tl.where(cand_offset==best,-1.0e9,candidate_scores)

        # Exact 256-local + eight selected remote token scores.
        local_offset=tl.arange(0,256)
        lp=i-127+local_offset
        lrange=(lp>=0)&(lp<T)
        li=tl.minimum(tl.maximum(lp,0),T-1)
        lvld=tl.load(VALID+b*T+li,mask=lrange,other=0)!=0
        lvalid=lrange&lvld
        lk=tl.load(
            K+((b*H+h)*T+li[:,None])*D+d[None,:],
            mask=lvalid[:,None]&(d[None,:]<D),other=0
        ).to(tl.float32)
        ls=tl.sum(lk*q[None,:],1)/tl.sqrt(tl.full((),D,tl.float32))
        rvld=tl.load(VALID+b*T+remote_idx,mask=remote_ok,other=0)!=0
        rvalid=remote_ok&rvld
        rk=tl.load(
            K+((b*H+h)*T+remote_idx[:,None])*D+d[None,:],
            mask=rvalid[:,None]&(d[None,:]<D),other=0
        ).to(tl.float32)
        rs=tl.sum(rk*q[None,:],1)/tl.sqrt(tl.full((),D,tl.float32))
        if HAS_MASK4:
            lm=tl.load(MASK+b*M0+i*M2+li*M3,mask=lrange,other=0)
            rm=tl.load(MASK+b*M0+i*M2+remote_idx*M3,mask=remote_ok,other=0)
            if MASK_IS_BOOL:
                lvalid=lvalid&(lm!=0)
                rvalid=rvalid&(rm!=0)
            else:
                ls=ls+lm.to(tl.float32)
                rs=rs+rm.to(tl.float32)
                lvalid=lvalid&(lm.to(tl.float32)>-1.0e4)
                rvalid=rvalid&(rm.to(tl.float32)>-1.0e4)
        ls=tl.where(lvalid,ls,-1.0e9)
        rs=tl.where(rvalid,rs,-1.0e9)
        maxscore=tl.maximum(tl.max(ls,0),tl.max(rs,0))
        le=tl.exp(ls-maxscore)
        re=tl.exp(rs-maxscore)
        denom=tl.sum(le,0)+tl.sum(re,0)
        lvv=tl.load(
            V+((b*H+h)*T+li[:,None])*D+d[None,:],
            mask=lvalid[:,None]&(d[None,:]<D),other=0
        ).to(tl.float32)
        rvv=tl.load(
            V+((b*H+h)*T+remote_idx[:,None])*D+d[None,:],
            mask=rvalid[:,None]&(d[None,:]<D),other=0
        ).to(tl.float32)
        numer=tl.sum(lvv*le[:,None],0)+tl.sum(rvv*re[:,None],0)
        out=numer/tl.maximum(denom,1.0e-12)
        tl.store(out_ptr,out,mask=d<D)
    else:
        tl.store(out_ptr,tl.full((BD,),0,tl.float32),mask=d<D)


def fused_triton_local256_remote8(q,k,v,valid,attention_mask=None):
    """No Python query loop. Two compiled Triton kernel launches per forward."""
    B,H,T,D=q.shape
    assert q.is_cuda and k.is_cuda and v.is_cuda
    assert BLOCK_SIZE==16 and ROUTE_BLOCKS==8
    assert q.shape==k.shape==v.shape and valid.shape==(B,T)
    assert q.is_contiguous() and k.is_contiguous() and v.is_contiguous()
    assert q.dtype==k.dtype==v.dtype
    valid=valid.to(torch.uint8).contiguous()
    if attention_mask is not None and attention_mask.ndim not in (2,4):
        raise ValueError(f"Unsupported native attention mask {attention_mask.shape}")
    mask4=attention_mask is not None and attention_mask.ndim==4
    if mask4 and attention_mask.shape[1]!=1:
        raise ValueError("V3.5 full-attention kernel needs [B,1,T,T] mask.")
    if mask4 and (attention_mask.shape[-2]!=T or attention_mask.shape[-1]!=T):
        raise ValueError("Unexpected full-attention mask shape.")
    # Native model uses one 4D key mask for all heads (or a 2D padding mask).
    mask=torch.empty((1,),dtype=torch.int32,device=q.device) if attention_mask is None else attention_mask
    strides=mask.stride() if mask4 else (0,0,0,0)
    G=triton.cdiv(T,16)
    BD=triton.next_power_of_2(D)
    BG=triton.next_power_of_2(G)
    proto=torch.empty((B,H,G,5,D),device=q.device,dtype=q.dtype)
    group_valid=torch.empty((B,G),device=q.device,dtype=torch.uint8)
    _build_key_prototypes[(G,H,B)](
        k,valid,proto,group_valid,
        T,H,D,G,BD,num_warps=4
    )
    result=torch.empty_like(q)
    _fused_local256_remote8[(T,H,B)](
        q,k,v,valid,proto,group_valid,mask,result,
        T,H,D,G,BD,BG,
        mask4, bool(mask4 and attention_mask.dtype==torch.bool),
        *strides,num_warps=8
    )
    return result

class FusedSparseFullAttention(nn.Module):
    """Full attention module replaced by two CUDA-compiled Triton kernels."""
    def __init__(self,original):
        super().__init__()
        self.config=original.config
        self.hidden_size=int(original.config.hidden_size)
        self.num_heads=int(original.num_heads)
        self.head_dim=int(original.head_dim)
        self.Wqkv=copy.deepcopy(original.Wqkv).requires_grad_(False)
        self.Wo=copy.deepcopy(original.Wo).requires_grad_(False)
        self.rotary_emb=copy.deepcopy(original.rotary_emb).requires_grad_(False)
        self.out_drop=copy.deepcopy(original.out_drop).requires_grad_(False)
        self.eval()

    def forward(self,hidden_states,attention_mask=None,position_ids=None,
                position_embeddings=None,sliding_window_mask=None,**kwargs):
        B,T,D=hidden_states.shape
        if position_ids is None:
            position_ids=torch.arange(T,device=hidden_states.device)[None,:]
        qkv=self.Wqkv(hidden_states).view(B,T,3,self.num_heads,self.head_dim)
        cos,sin=self.rotary_emb(qkv,position_ids=position_ids)
        q,k,v=qkv.transpose(3,1).unbind(dim=2)
        q,k=apply_rotary_pos_emb(q,k,cos,sin)
        # Triton expects [B,H,T,D] packed contiguous buffers.
        q=q.contiguous()
        k=k.contiguous()
        v=v.contiguous()
        valid=_valid_tokens(attention_mask,hidden_states)
        core=fused_triton_local256_remote8(q,k,v,valid,attention_mask)
        projected=self.Wo(
            core.transpose(1,2).contiguous().reshape(B,T,D)
        )
        return self.out_drop(projected),None

def build_fused_layer(index):
    return FusedSparseFullAttention(
        teacher.encoder.layers[index].attn
    ).to(device).eval().requires_grad_(False)
print("Compiled Triton kernels defined (first call triggers actual CUDA JIT).")


In [ ]:
#@title 6. CUDA compile, correctness preflight and padding invariance
# A real CUDA GPU must JIT-compile and run these kernels before screening begins.
def compare_core(q,k,v,valid,mask=None,expected=None):
    actual=fused_triton_local256_remote8(q.contiguous(),k.contiguous(),
                                         v.contiguous(),valid,mask)
    ref=expected if expected is not None else F.scaled_dot_product_attention(
        q,k,v,attn_mask=mask if mask is not None and mask.ndim==4
        else valid[:,None,None,:]
    )
    ref=ref*valid[:,None,:,None].to(ref.dtype)
    difference=float((actual.float()-ref.float()).abs().amax())
    cosine=float(F.cosine_similarity(
        actual.float()[valid[:,None,:].expand(q.shape[:3])],
        ref.float()[valid[:,None,:].expand(q.shape[:3])],dim=-1
    ).mean())
    return difference,cosine

# Short sequences have every valid key inside the 256-local window.
for t in (9,32,127,256):
    torch.manual_seed(2035+t)
    q=torch.randn((1,2,t,32),device=device,dtype=amp_dtype).contiguous()
    k=torch.randn_like(q).contiguous()
    v=torch.randn_like(q).contiguous()
    valid=torch.ones((1,t),device=device,dtype=torch.bool)
    valid[:,-max(1,t//9):]=False
    masks=[
        valid[:,None,None,:].expand(1,1,t,t).contiguous(),
        torch.where(valid[:,None,None,:].expand(1,1,t,t),
                    torch.tensor(0.,device=device),
                    torch.tensor(-10000.,device=device)).contiguous()
    ]
    for mask in masks:
        with torch.inference_mode():
            delta,sim=compare_core(q,k,v,valid,mask)
        assert delta<0.025 and sim>0.999,(t,str(mask.dtype),delta,sim)
        print(f"Triton short native parity T={t}, mask={str(mask.dtype)} "
              f"PASS maxΔ={delta:.6g} cos={sim:.7f}")
# Long sequence: exact comparison to the V3.4 unfused router, not oracle.
for t in (279,320,512):
    torch.manual_seed(3500+t)
    q=torch.randn((1,2,t,32),device=device,dtype=amp_dtype).contiguous()
    k=torch.randn_like(q).contiguous()
    v=torch.randn_like(q).contiguous()
    valid=torch.ones((1,t),device=device,dtype=torch.bool)
    valid[:,-min(41,t//8):]=False
    with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=amp_dtype):
        ref=python_reference_sparse_core(
            q,k,v,valid,valid.long(),
            mode="local256_remote8",block=16,
            route_blocks=8,query_chunk=QUERY_CHUNK
        )
        delta,sim=compare_core(q,k,v,valid,valid.long(),expected=ref)
    assert delta<0.075 and sim>0.998,(t,delta,sim)
    print(f"Triton vs unfused sparse T={t} PASS maxΔ={delta:.6g} cos={sim:.7f}")

# Real pretrained layer, same-length masked noise invariance.
teacher_index=full_layers[0]
target=build_fused_layer(teacher_index)
x=torch.randn((1,320,target.hidden_size),device=device,dtype=torch.float32)
valid=torch.zeros((1,320),device=device,dtype=torch.long)
valid[:,:279]=1
zero=x.clone();zero[:,279:]=0
noise=x.clone();noise[:,279:]=torch.randn_like(noise[:,279:])*6
with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=amp_dtype):
    a=target(zero,attention_mask=valid)[0][:,:279].float()
    b=target(noise,attention_mask=valid)[0][:,:279].float()
delta=float((a-b).abs().amax())
assert delta<0.02,("Padding leaked into valid tokens",delta)
print("Real full-layer Triton masking PASS maxΔ",delta)
# Native pretrained full-attention replay, using actual encoder arguments.
sample=attn_io(teacher,teacher_index,batch(probe_items,0,BATCH))
with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=amp_dtype):
    out=_attention_output_tensor(
        teacher.encoder.layers[teacher_index].attn(
            *sample["call_args"],**sample["call_kwargs"]
        )
    )
replay=float((out.float()-sample["y"].float()).abs().amax())
assert replay<1e-2,replay
print("Native teacher SDPA exact replay: PASS",replay)
del target
torch.cuda.empty_cache()


## Protocol: isolate fidelity, then replace every full-attention layer

**This is the same zero-shot frozen pretrained architecture as V3.4**. The numerical preflight compares short-sequence attention with native SDPA and longer sequences with the actual V3.4 Python reference. The fitted/pretrained weights are unchanged.

Evaluation results always name their method: \`native_sdpa\`, \`python_sparse_v34\`, \`fused_triton_sparse_v35\`, or \`adaptive_native_fallback\`. No inference path uses teacher-oracle keys. For short sequences (≤256), the fused kernel's local window already contains all valid keys, so it is mathematically equivalent to full attention (apart from numerical precision).

GPU latency is measured after Triton JIT compilation and warm-up. For fair comparison, cache the frozen teacher's input tensors for each layer and measure the original native module with its **exact** captured kwargs. **End-to-end** accuracy and latency are evaluated with *all* full-attention modules replaced, leaving sliding modules untouched.

An optional deployment-oriented adaptive variant keeps native attention at layers where dense SDPA wins timing. This is evaluated separately, and **does not count** as a complete sparse-layer replacement.


In [ ]:
#@title 7. Validation helpers and matched CUDA wall-time benchmarking
@torch.inference_mode()
def check_decisions(model,items,limit=AUDIT_DECISIONS):
    n=matched=teacher_gold=student_gold=0
    total_js=0.0
    end=min(len(items),int(limit))
    for offset in range(0,end,BATCH):
        chosen=items[offset:min(offset+BATCH,end)]
        b=direct_batch(chosen)
        with torch.autocast(device_type="cuda",dtype=amp_dtype):
            tlog,_=teacher(**b)
            slog,_=model(**b)
        require_finite(slog,"V3.5 full-model logits")
        allowed=b["marker_mask"].bool()
        pt=torch.softmax(tlog.float().masked_fill(~allowed,-1e9),dim=-1)
        ps=torch.softmax(slog.float().masked_fill(~allowed,-1e9),dim=-1)
        tpred=pt.argmax(-1);spred=ps.argmax(-1)
        gold=torch.tensor([c["gold_index"] for c in chosen],device=device)
        n+=len(chosen)
        matched+=int((tpred==spred).sum())
        teacher_gold+=int((tpred==gold).sum())
        student_gold+=int((spred==gold).sum())
        mean=0.5*(pt+ps)
        js=0.5*((pt*(pt.clamp_min(1e-8).log()-mean.clamp_min(1e-8).log())).sum(-1)
               +(ps*(ps.clamp_min(1e-8).log()-mean.clamp_min(1e-8).log())).sum(-1))
        total_js+=float(js.sum())
    return dict(decisions=n,teacher_agreement=matched/max(n,1),
                teacher_gold_accuracy=teacher_gold/max(n,1),
                student_gold_accuracy=student_gold/max(n,1),
                gold_accuracy_drop=(teacher_gold-student_gold)/max(n,1),
                mean_js=total_js/max(n,1))

@torch.inference_mode()
def benchmark_cuda(fn,warmup=4,repeats=12):
    with torch.autocast(device_type="cuda",dtype=amp_dtype):
        for _ in range(warmup): fn()
        torch.cuda.synchronize()
        ticks=[]
        for _ in range(repeats):
            start=time.perf_counter()
            fn()
            torch.cuda.synchronize()
            ticks.append((time.perf_counter()-start)*1000)
    return float(np.median(ticks))

def layer_local_stats(replacement,cache):
    # Same frozen teacher layer x/y on valid tokens. No oracle-key routing.
    replacement.eval()
    total_error=total_power=total_cos=0.0
    count=0
    with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=amp_dtype):
        for entry in cache:
            y,_=replacement(
                entry["x"],attention_mask=entry["mask"],
                position_ids=entry["call_kwargs"].get("position_ids")
            )
            valid=entry["valid"]
            z=y.float()[valid]
            truth=entry["y"].float()[valid]
            total_error+=float((z-truth).square().sum())
            total_power+=float(truth.square().sum())
            total_cos+=float(F.cosine_similarity(z,truth,dim=-1).sum())
            count+=int(valid.sum())
    return dict(cosine=total_cos/max(1,count),
                nmse=total_error/max(1e-8,total_power),
                valid_tokens=count)

def strict_native_layer_replay(idx,entry):
    native=teacher.encoder.layers[idx].attn
    with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=amp_dtype):
        y=_attention_output_tensor(native(
            *entry["call_args"],**entry["call_kwargs"]
        ))
    maxdiff=float((y.float()-entry["y"].float()).abs().amax())
    assert maxdiff<1e-2,(idx,maxdiff)
    return maxdiff

print("V3.5 per-layer indices:",full_layers)
print("Untouched model baseline:",check_decisions(student_eval,audit_items,limit=AUDIT_DECISIONS))


In [ ]:
#@title 8. All 10 full-attention layers: frozen isolated fidelity and matched latency
# For a fast evaluation start with 2 probe caches and 3 audit caches per layer.
# All layer timings include QKV, RoPE, 2 Triton kernels and Wo.
per_layer={}
for idx in full_layers:
    cache=cache_teacher_io(idx,audit_items,AUDIT_CACHE_BATCHES)
    entry=cache[0]
    strict_native_layer_replay(idx,entry)
    layer=build_fused_layer(idx)
    quality=layer_local_stats(layer,cache)
    original=teacher.encoder.layers[idx].attn
    native_ms=benchmark_cuda(
        lambda:_attention_output_tensor(
            original(*entry["call_args"],**entry["call_kwargs"])
        ),warmup=3,repeats=10
    )
    fused_ms=benchmark_cuda(
        lambda:layer(
            entry["x"],attention_mask=entry["mask"],
            position_ids=entry["call_kwargs"].get("position_ids")
        ),warmup=3,repeats=10
    )
    python_ms=None
    python_cosine=None
    if BENCHMARK_PYTHON_REFERENCE:
        # This is a REFERENCE, not part of the fast inference path.
        B,T,D=entry["x"].shape
        H,hd=original.num_heads,original.head_dim
        with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=amp_dtype):
            qkv=original.Wqkv(entry["x"]).view(B,T,3,H,hd)
            cos,sin=original.rotary_emb(qkv,
                position_ids=entry["call_kwargs"].get("position_ids"))
            q,k,v=qkv.transpose(3,1).unbind(dim=2)
            q,k=apply_rotary_pos_emb(q,k,cos,sin)
            q=q.contiguous();k=k.contiguous();v=v.contiguous()
        ref_valid=entry["valid"]
        python_ms=benchmark_cuda(
            lambda:python_reference_sparse_core(
                q,k,v,ref_valid,entry["mask"],mode="local256_remote8",
                block=16,route_blocks=8,query_chunk=QUERY_CHUNK
            ),warmup=1,repeats=4
        )
        # Compare the actual pretrained-frozen sparse core with the compiled core.
        with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=amp_dtype):
            core_ref=python_reference_sparse_core(
                q,k,v,ref_valid,entry["mask"],mode="local256_remote8",
                block=16,route_blocks=8,query_chunk=QUERY_CHUNK
            )
            core_fast=fused_triton_local256_remote8(q,k,v,ref_valid,entry["mask"])
        valid_core=ref_valid[:,None,:].expand(B,H,T)
        python_cosine=float(F.cosine_similarity(
            core_fast.float()[valid_core],core_ref.float()[valid_core],dim=-1).mean())
        del qkv,cos,sin,q,k,v,core_ref,core_fast
    per_layer[str(idx)]=dict(
        layer=idx,quality=quality,
        native_ms=native_ms,
        v34_python_core_ms=python_ms,
        v35_fused_layer_ms=fused_ms,
        fused_vs_python_core_cosine=python_cosine,
        speedup_over_native=native_ms/max(fused_ms,1e-8)
    )
    print(f"L{idx:02d}: cosine={quality['cosine']:.5f} "
          f"NMSE={quality['nmse']:.5f}, "
          f"native={native_ms:.3f} ms, Triton={fused_ms:.3f} ms "
          f"native/Triton={native_ms/max(fused_ms,1e-8):.3f}x "
          f"| Python core={python_ms}ms")
    del layer,cache
    torch.cuda.empty_cache()


In [ ]:
#@title 9. Replace EVERY full-attention module simultaneously, benchmark complete model
all_fused_metrics=None
all_fused_timing=None
cumulative=[]
if RUN_ALL_FULL_LAYERS:
    originals={idx:student_eval.encoder.layers[idx].attn for idx in full_layers}
    try:
        for idx in full_layers:
            student_eval.encoder.layers[idx].attn=build_fused_layer(idx)
            stats=check_decisions(
                student_eval,audit_items,limit=CUMULATIVE_E2E_DECISIONS
            )
            cumulative.append(dict(layer=idx,metrics=stats))
            print(f"Installed {idx}: agree={stats['teacher_agreement']:.4f}, "
                  f"gold={stats['student_gold_accuracy']:.4f}, JS={stats['mean_js']:.6f}")
        all_fused_metrics=check_decisions(student_eval,audit_items,limit=AUDIT_DECISIONS)
        sample_b=batch(audit_items,0,BATCH)
        native_ms=benchmark_cuda(lambda:teacher(**sample_b),warmup=3,repeats=8)
        fused_ms=benchmark_cuda(lambda:student_eval(**sample_b),warmup=3,repeats=8)
        all_fused_timing=dict(
            native_ms=native_ms,fused_triton_ms=fused_ms,
            speedup=native_ms/max(fused_ms,1e-8)
        )
        print("\nALL FULL ATTENTION REPLACED:",len(full_layers),"of",len(full_layers))
        print("All-fused model accuracy:",all_fused_metrics)
        print("All-fused whole-model latency:",all_fused_timing)
    finally:
        for idx,native in originals.items():
            student_eval.encoder.layers[idx].attn=native
        student_eval.eval().requires_grad_(False)
        torch.cuda.empty_cache()
else:
    print("RUN_ALL_FULL_LAYERS=False; no cumulative replacement.")


In [ ]:
#@title 10. OPTIONAL speed-based native fallback, evaluated separately (not all-sparse)
adaptive_metrics=None
adaptive_timing=None
adaptive_routes=None
if RUN_NATIVE_FALLBACK_COMPARISON:
    originals={idx:student_eval.encoder.layers[idx].attn for idx in full_layers}
    adaptive_routes={}
    try:
        for idx in full_layers:
            native_time=per_layer[str(idx)]["native_ms"]
            fused_time=per_layer[str(idx)]["v35_fused_layer_ms"]
            if fused_time < native_time and per_layer[str(idx)]["quality"]["cosine"]>=0.95:
                student_eval.encoder.layers[idx].attn=build_fused_layer(idx)
                adaptive_routes[str(idx)]="fused_triton"
            else:
                adaptive_routes[str(idx)]="native_sdpa"
        adaptive_metrics=check_decisions(
            student_eval,audit_items,limit=AUDIT_DECISIONS
        )
        timed=batch(audit_items,0,BATCH)
        adaptive_ms=benchmark_cuda(lambda:student_eval(**timed),warmup=3,repeats=8)
        native_ms=benchmark_cuda(lambda:teacher(**timed),warmup=3,repeats=8)
        adaptive_timing=dict(native_ms=native_ms,adaptive_ms=adaptive_ms,
                             speedup=native_ms/max(adaptive_ms,1e-8))
        print("Adaptive choices:",adaptive_routes)
        print("Adaptive (may retain native layers) quality:",adaptive_metrics)
        print("Adaptive latency:",adaptive_timing)
    finally:
        for idx,native in originals.items():
            student_eval.encoder.layers[idx].attn=native
        student_eval.eval().requires_grad_(False)
        torch.cuda.empty_cache()


In [ ]:
#@title 11. Full report, stop/go thresholds and GPU speedup comparison
if not per_layer:
    raise RuntimeError("Run the per-layer audit first.")
min_cos=min(row["quality"]["cosine"] for row in per_layer.values())
layer_ok=min_cos>=.95
decision_ok=all_fused_metrics is not None and all_fused_metrics["teacher_agreement"]>=.95
gold_ok=all_fused_metrics is not None and all_fused_metrics["gold_accuracy_drop"]<=.01
speed_ok=all_fused_timing is not None and all_fused_timing["speedup"]>1.0
passed=bool(layer_ok and decision_ok and gold_ok and speed_ok)
report=dict(
    version="3.5-fused-triton-cuda-all-full-attention",
    model=SOURCE_MODEL,seed=SEED,gpu=torch.cuda.get_device_name(0),
    torch=torch.__version__,triton=triton.__version__,
    official_test_used=False,
    full_attention_layer_indices=full_layers,
    full_layers_requested=len(full_layers),
    full_layers_replaced=len(full_layers) if all_fused_metrics is not None else 0,
    architecture="pretrained RoPE Q/K/V, per-head five-prototype block router,"
                 " local-256+query-routed remote-8, fused sparse CUDA attention",
    kernel_compilation="Triton CUDA JIT",
    routing_blocks=8,block_size=16,local_window=256,remote_tokens=8,
    retrieval_training_steps=0,
    isolated=per_layer,cumulative=cumulative,
    all_fused_metrics=all_fused_metrics,all_fused_timing=all_fused_timing,
    optional_adaptive_native_fallback=dict(
        choices=adaptive_routes,metrics=adaptive_metrics,
        timing=adaptive_timing
    ),
    gates=dict(all_layer_cosine_ge_095=layer_ok,
               full_teacher_agreement_ge_095=decision_ok,
               gold_accuracy_drop_le_1pp=gold_ok,
               end_to_end_faster_than_native=speed_ok,
               overall_pass=passed),
    warnings=[
        "V3.5 sparse attention is NOT an attention-free CeNN.",
        "Triton kernels use actual heuristic query-to-block selection, not the V3.3 oracle.",
        "Kernel compile latency excluded from warmed inference timing.",
        "All-fused model and optional native-fallback model are distinct benchmarks.",
        "Fewer score calculations do not guarantee faster GPU latency on 512-token contexts.",
        "Matched evaluation uses a small held-out case-disjoint subset of train; official test untouched.",
        "No checkpoint is overwritten or uploaded to Hugging Face.",
    ]
)
path=OUTPUT_DIR/"v35_all_full_triton_cuda_report.json"
path.write_text(json.dumps(report,indent=2,allow_nan=False),encoding="utf-8")
print("Saved report:",path)
print("Final gates:",report["gates"])
if passed:
    print("PASS: check additional lengths, batches and seeds before deployment.")
else:
    print("NOT READY for replacement: compare fidelity, cumulative decision loss and GPU speed.")
import matplotlib.pyplot as plt
rows=[per_layer[str(k)] for k in full_layers]
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(full_layers,[r["v35_fused_layer_ms"] for r in rows],"o-",label="V3.5 Triton")
ax.plot(full_layers,[r["native_ms"] for r in rows],"o-",label="Native SDPA")
ax.set_xlabel("Full-attention layer")
ax.set_ylabel("Layer latency (ms), lower is better")
ax.set_title("Pretrained full-attention layer CUDA latency")
ax.grid(alpha=.3);ax.legend();plt.show()
